# Debug Individual Event Training

Run individual events through the Cox model pipeline to diagnose NaN issues.
Tests data quality and model fitting for specific events across **all schemes** (death_met, icd3, icd4, phecode).

In [1]:
import sys
import os
import logging
import numpy as np
import pandas as pd

# Add project paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'python_utils'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'python_scripts', 'model_training'))

from embed_surv_utils import run_base_CoxPH, run_grid_CoxPH_parallel
from slurm_array_utils import (
    SCHEME_CONFIG, DEFAULT_ALPHAS, DEFAULT_L1_RATIOS, MET_EVENTS,
    build_full_prediction_df, filter_event_rows,
)

ALL_SCHEMES = sorted(SCHEME_CONFIG.keys())

# Enable logging so we can see warnings from cox_models
logging.basicConfig(level=logging.INFO, format='%(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger('embed_surv_utils.cox_models')
logger.setLevel(logging.DEBUG)

## 1. Load all schemes and inspect available events

In [2]:
scheme_data = {}
for scheme in ALL_SCHEMES:
    print(f"\n--- Loading scheme: {scheme} ---")
    full_prediction_df, type_cols, embed_cols, events = build_full_prediction_df(scheme)
    scheme_data[scheme] = {
        'df': full_prediction_df,
        'type_cols': type_cols,
        'embed_cols': embed_cols,
        'events': events,
    }
    print(f"  Shape: {full_prediction_df.shape}")
    print(f"  Events: {len(events)}")
    print(f"  Embedding cols: {len(embed_cols)}")

print(f"\nLoaded {len(scheme_data)} schemes: {list(scheme_data.keys())}")
for scheme, data in scheme_data.items():
    print(f"  {scheme}: {len(data['events'])} events, {data['df'].shape[0]} patients")


--- Loading scheme: death_met ---
  Shape: (25162, 2341)
  Events: 8
  Embedding cols: 2306

--- Loading scheme: icd3 ---
  Shape: (25162, 2533)
  Events: 104
  Embedding cols: 2306

--- Loading scheme: icd4 ---
  Shape: (25162, 2503)
  Events: 89
  Embedding cols: 2306

--- Loading scheme: phecode ---
  Shape: (25162, 2493)
  Events: 84
  Embedding cols: 2306

Loaded 4 schemes: ['death_met', 'icd3', 'icd4', 'phecode']
  death_met: 8 events, 25162 patients
  icd3: 104 events, 25162 patients
  icd4: 89 events, 25162 patients
  phecode: 84 events, 25162 patients


## 2. Data quality diagnostics per scheme

Check NaN counts, negative times, event rates, and sample sizes.

In [3]:
# Pick a scheme to inspect in detail (change as needed)
SCHEME = 'death_met'  # 'death_met', 'icd3', 'icd4', 'phecode'

sd = scheme_data[SCHEME]
full_prediction_df = sd['df']
type_cols = sd['type_cols']
embed_cols = sd['embed_cols']
events = sd['events']

# For death_met, show all events; for ICD/phecode, show first 10
if SCHEME == 'death_met':
    debug_events = events
else:
    debug_events = events[:10]
print(f"Scheme: {SCHEME}, debug events ({len(debug_events)}/{len(events)}): {debug_events}")

Scheme: death_met, debug events (8/8): ['death', 'adrenalM', 'boneM', 'brainM', 'liverM', 'lungM', 'nodeM', 'peritonealM']


In [4]:
diag_rows = []
for event in debug_events:
    tt_col = f'tt_{event}'
    tt = full_prediction_df[tt_col]
    ev = full_prediction_df[event]
    
    row = {
        'event': event,
        'n_total': len(tt),
        'tt_nan': tt.isna().sum(),
        'tt_negative': (tt < 0).sum(),
        'tt_zero': (tt == 0).sum(),
        'tt_positive': (tt > 0).sum(),
        'event_nan': ev.isna().sum(),
        'event_rate': ev.mean(),
        'n_events': ev.sum(),
        'tt_min': tt.min(),
        'tt_max': tt.max(),
        'tt_median': tt.median(),
    }
    diag_rows.append(row)

diag_df = pd.DataFrame(diag_rows)
diag_df

,event,n_total,tt_nan,tt_negative,tt_zero,tt_positive,event_nan,event_rate,n_events,tt_min,tt_max,tt_median
0,death,25162,0,0,0,25162,0,0.491416,12365,1.0,3625.0,897.5
1,adrenalM,25162,0,270,15,24877,0,0.046141,1161,-2624.0,3625.0,863.0
2,boneM,25162,0,1139,61,23962,0,0.160202,4031,-3881.0,3625.0,755.0
3,brainM,25162,0,818,33,24311,0,0.099873,2513,-3341.0,3625.0,821.0
4,liverM,25162,0,1153,47,23962,0,0.162269,4083,-2765.0,3625.0,762.0
5,lungM,25162,0,1515,76,23571,0,0.193069,4858,-3881.0,3625.0,712.0
6,nodeM,25162,0,1762,82,23318,0,0.210516,5297,-3881.0,3625.0,675.5
7,peritonealM,25162,0,613,20,24529,0,0.100787,2536,-3938.0,3625.0,818.0


In [5]:
# Check for NaN in embedding columns
embed_nan_per_col = full_prediction_df[embed_cols].isna().sum()
n_embed_nan = embed_nan_per_col.sum()
print(f"Total NaN values across {len(embed_cols)} embedding columns: {n_embed_nan}")
if n_embed_nan > 0:
    print("\nColumns with NaN:")
    print(embed_nan_per_col[embed_nan_per_col > 0])

# Check for NaN in base covariates
base_vars = ['GENDER', 'AGE_AT_TREATMENTSTART']
for col in base_vars + type_cols:
    n_nan = full_prediction_df[col].isna().sum()
    if n_nan > 0:
        print(f"  {col}: {n_nan} NaN values")

Total NaN values across 2306 embedding columns: 0


## 3. Test individual events through the pipeline

Run `run_base_CoxPH` and `run_grid_CoxPH_parallel` on individual events to see which ones fail and what errors are raised.

In [6]:
base_vars = ['GENDER', 'AGE_AT_TREATMENTSTART']

def run_single_event_debug(event, scheme_df, type_cols, embed_cols, run_grid=True):
    """Run both base and grid models for a single event, printing diagnostics."""
    tt_col = f'tt_{event}'
    print(f"\n{'='*60}")
    print(f"EVENT: {event}")
    print(f"{'='*60}")
    
    # Filter rows
    event_df = filter_event_rows(scheme_df, event)
    print(f"Rows after filtering: {len(event_df)} (from {len(scheme_df)})")
    print(f"Event rate: {event_df[event].mean():.4f} ({int(event_df[event].sum())} events)")
    print(f"Time range: [{event_df[tt_col].min():.1f}, {event_df[tt_col].max():.1f}] days")
    
    if event_df.empty:
        print("  -> SKIPPED: no valid rows")
        return None, None
    
    # Check feature matrix for NaN/inf
    all_feature_cols = base_vars + type_cols + embed_cols
    X_check = event_df[all_feature_cols].to_numpy(dtype=np.float32)
    print(f"Feature matrix: {X_check.shape}")
    print(f"  NaN count: {np.isnan(X_check).sum()}")
    print(f"  Inf count: {np.isinf(X_check).sum()}")
    
    # Zero-variance columns
    stds = np.std(X_check, axis=0)
    zero_var_cols = [all_feature_cols[i] for i in range(len(stds)) if stds[i] == 0]
    if zero_var_cols:
        print(f"  Zero-variance columns ({len(zero_var_cols)}): {zero_var_cols[:10]}...")
    
    # Run baseline CoxPH
    print("\n--- Baseline CoxPH ---")
    try:
        base_results = run_base_CoxPH(
            event_df, base_vars + type_cols, ['AGE_AT_TREATMENTSTART'],
            event_col=event, tstop_col=tt_col,
        )
        print(base_results.to_string(index=False))
        has_nan = base_results[['mean_c_index', 'mean_auc(t)', 'mean_ibs']].isna().any().any()
        if has_nan:
            print("  ** WARNING: NaN in baseline results **")
    except Exception as e:
        print(f"  ** FAILED: {e} **")
        base_results = None
    
    # Run grid search (text embeddings)
    grid_test = None
    if run_grid:
        print("\n--- Grid CoxPH (text embeddings) ---")
        try:
            grid_test, grid_val, _ = run_grid_CoxPH_parallel(
                event_df, base_vars + type_cols,
                ['AGE_AT_TREATMENTSTART'] + embed_cols,
                embed_cols,
                DEFAULT_L1_RATIOS, DEFAULT_ALPHAS,
                event_col=event, tstop_col=tt_col,
                max_iter=1000, n_jobs=1,  # single-threaded for debugging
            )
            print("Test metrics:")
            print(grid_test.to_string(index=False))
            
            # Show best CV result
            valid_cv = grid_val.dropna(subset=['mean_auc(t)'])
            if not valid_cv.empty:
                best = valid_cv.sort_values('mean_auc(t)', ascending=False).iloc[0]
                print(f"Best CV: alpha={best['alpha']:.2e}, l1={best['l1_ratio']:.1f}, "
                      f"AUC(t)={best['mean_auc(t)']:.4f}, error_rate={best['error_rate']:.2f}")
            else:
                print("  ** WARNING: ALL CV results are NaN **")
            
            has_nan = grid_test.isna().any().any()
            if has_nan:
                print("  ** WARNING: NaN in grid search test results **")
        except Exception as e:
            print(f"  ** FAILED: {e} **")
    
    return base_results, grid_test

### 3a. Test death event

In [7]:
if 'death' in events:
    base_death, grid_death = run_single_event_debug(
        'death', full_prediction_df, type_cols, embed_cols, run_grid=True
    )
else:
    print("'death' event not found in this scheme. Available events:", events[:10])


EVENT: death
Rows after filtering: 25162 (from 25162)
Event rate: 0.4914 (12365 events)
Time range: [1.0, 3625.0] days
Feature matrix: (25162, 2322)
  NaN count: 0
  Inf count: 0

--- Baseline CoxPH ---
eval_data  mean_c_index  mean_auc(t)  mean_ibs
  cv_data      0.643020     0.678719  0.197405
test_data      0.646773     0.684172  0.199615

--- Grid CoxPH (text embeddings) ---
Test metrics:
 mean_auc(t)  mean_ibs  mean_c_index
    0.765233  0.179463      0.722638
Best CV: alpha=7.50e-04, l1=0.5, AUC(t)=0.7599, error_rate=0.00


### 3b. Test metastasis events

In [ ]:
met_events_present = [e for e in events if e in MET_EVENTS]
print(f"Metastasis events in this scheme: {met_events_present}")

met_results = {}
for met_event in met_events_present:
    base_met, grid_met = run_single_event_debug(
        met_event, full_prediction_df, type_cols, embed_cols, run_grid=True
    )
    met_results[met_event] = {'base': base_met, 'grid': grid_met}

Metastasis events in this scheme: ['adrenalM', 'boneM', 'brainM', 'liverM', 'lungM', 'nodeM', 'peritonealM']

EVENT: adrenalM
Rows after filtering: 24877 (from 25162)
Event rate: 0.0352 (876 events)
Time range: [1.0, 3625.0] days
Feature matrix: (24877, 2322)
  NaN count: 0
  Inf count: 0

--- Baseline CoxPH ---
eval_data  mean_c_index  mean_auc(t)  mean_ibs
  cv_data      0.750991     0.773840  0.037494
test_data      0.761637     0.781944  0.036767

--- Grid CoxPH (text embeddings) ---


## 4. Summary of results across tested events

In [ ]:
summary_rows = []

# Collect all tested events
all_tested = {}
if 'death' in events:
    all_tested['death'] = {'base': base_death, 'grid': grid_death}
all_tested.update(met_results)

for event_name, results in all_tested.items():
    row = {'event': event_name}
    
    if results['base'] is not None:
        cv_row = results['base'][results['base']['eval_data'] == 'cv_data']
        test_row = results['base'][results['base']['eval_data'] == 'test_data']
        row['base_cv_auc'] = cv_row['mean_auc(t)'].values[0] if len(cv_row) else np.nan
        row['base_test_auc'] = test_row['mean_auc(t)'].values[0] if len(test_row) else np.nan
    else:
        row['base_cv_auc'] = 'FAILED'
        row['base_test_auc'] = 'FAILED'
    
    if results['grid'] is not None:
        row['grid_test_auc'] = results['grid']['mean_auc(t)'].values[0]
        row['grid_test_cindex'] = results['grid']['mean_c_index'].values[0]
    else:
        row['grid_test_auc'] = 'FAILED'
        row['grid_test_cindex'] = 'FAILED'
    
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print("\nSummary of all tested events:")
summary_df

## 5. Baseline scan across ALL schemes and events

Run the baseline CoxPH (fast, no embeddings) across every event in every scheme to find which ones produce NaN.

In [ ]:
base_vars = ['GENDER', 'AGE_AT_TREATMENTSTART']
scan_rows = []

for scheme, sd in scheme_data.items():
    df = sd['df']
    tc = sd['type_cols']
    evts = sd['events']
    print(f"\nScanning {scheme}: {len(evts)} events...")
    
    for event in evts:
        tt_col = f'tt_{event}'
        event_df = filter_event_rows(df, event)
        if event_df.empty:
            scan_rows.append({'scheme': scheme, 'event': event, 'n_rows': 0, 'n_events': 0,
                              'event_rate': '', 'cv_auc': 'EMPTY', 'test_auc': 'EMPTY'})
            continue
        
        try:
            base_res = run_base_CoxPH(
                event_df, base_vars + tc, ['AGE_AT_TREATMENTSTART'],
                event_col=event, tstop_col=tt_col,
            )
            cv_auc = base_res.loc[base_res['eval_data'] == 'cv_data', 'mean_auc(t)'].values[0]
            test_auc = base_res.loc[base_res['eval_data'] == 'test_data', 'mean_auc(t)'].values[0]
        except Exception as e:
            cv_auc = f'ERROR: {e}'
            test_auc = f'ERROR: {e}'
        
        scan_rows.append({
            'scheme': scheme,
            'event': event,
            'n_rows': len(event_df),
            'n_events': int(event_df[event].sum()),
            'event_rate': f"{event_df[event].mean():.4f}",
            'cv_auc': cv_auc,
            'test_auc': test_auc,
        })

scan_df = pd.DataFrame(scan_rows)
print(f"\nTotal: scanned {len(scan_rows)} scheme-event combinations across {len(scheme_data)} schemes")
scan_df.groupby('scheme').size()

In [ ]:
# Show events that produced NaN or errors, grouped by scheme
nan_events = scan_df[
    scan_df['cv_auc'].apply(lambda x: isinstance(x, float) and np.isnan(x))
    | scan_df['test_auc'].apply(lambda x: isinstance(x, float) and np.isnan(x))
    | scan_df['cv_auc'].apply(lambda x: isinstance(x, str) and x.startswith('ERROR'))
    | scan_df['cv_auc'].apply(lambda x: isinstance(x, str) and x == 'EMPTY')
]

for scheme in ALL_SCHEMES:
    scheme_nan = nan_events[nan_events['scheme'] == scheme]
    scheme_total = scan_df[scan_df['scheme'] == scheme]
    print(f"\n{scheme}: {len(scheme_nan)}/{len(scheme_total)} events with NaN/errors/empty")
    if not scheme_nan.empty:
        display(scheme_nan)

# Show successful events summary
ok_events = scan_df[~scan_df.index.isin(nan_events.index)]
print(f"\n--- Successfully trained: {len(ok_events)}/{len(scan_df)} events ---")
ok_events.groupby('scheme').size()